Showcase the categorization of customer feedback using fine-tuned GPT-2.

We have a dataset of customer feedback that we want to categorize into different categories. We will use the fine-tuned GPT-2 model to classify the feedback into different categories.

In [1]:
# Load datasets
from datasets import load_dataset

from pathlib import Path

notebook_path = Path.cwd()

# Save synthetic data to CSV
train_path = notebook_path / "synthetic_train_data.csv"
test_path = notebook_path / "synthetic_test_data.csv"

dataset = load_dataset("csv", data_files={"train": str(train_path.absolute()), "test": str(test_path.absolute())})

We need to load the fine-tuned GPT-2 model and the dataset of customer feedback. We will then use the model to predict the category of each feedback.

We start this by loading the tokenizer and the model.

In [2]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load the tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Add a padding token to the tokenizer
tokenizer.add_special_tokens({'pad_token': '[PAD]'})
model.resize_token_embeddings(len(tokenizer))

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Tokenize the training and testing datasets
tokenized_datasets = dataset.map(tokenize_function, batched=True)

/home/amruthvvkp/projects/ai-playground/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Now we need to fine-tune the model on the dataset of customer feedback. We will use the fine-tuned model to predict the category of each feedback.

We will then evaluate the performance of the model by comparing the predicted categories with the actual categories of the feedback.

Finally, we will use the model to predict the category of new customer feedback.

In [3]:
import numpy as np
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Fine-tune the model
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
)

# Use a data collator for causal language modeling (not MLM, as GPT-2 is autoregressive)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Turn off MLM as this is a causal language model
)

# Ensure that labels are passed for loss calculation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": (predictions == labels).mean()}

# Use the Trainer for training with labels (causal language modeling)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,  # Pass the causal language modeling collator
    compute_metrics=compute_metrics,
)

# Start training
trainer.train()


/home/amruthvvkp/projects/ai-playground/.venv/lib/python3.12/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,83.904442,0.000000
2,No log,76.844688,0.000000
3,No log,73.991501,0.000000


TrainOutput(global_step=9, training_loss=78.51497395833333, metrics={'train_runtime': 620.7252, 'train_samples_per_second': 0.048, 'train_steps_per_second': 0.014, 'total_flos': 15677521920000.0, 'train_loss': 78.51497395833333, 'epoch': 3.0})

We need to define a preprocessing function to clean the text data and convert it into a format that the model can understand. We also need to define a function to predict the category of the feedback using the fine-tuned GPT-2 model.

We are trying to use Langchain to predict the category of customer feedback using fine-tuned GPT-2. We will use the fine-tuned model to classify the feedback into different categories.

In [7]:
import pandas as pd
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from sklearn.preprocessing import MinMaxScaler
from langchain.tools import BaseTool
from typing import Optional, Union, List
from langchain.callbacks.manager import CallbackManagerForToolRun, AsyncCallbackManagerForToolRun
from typing import ClassVar
class DataPreprocessingTool(BaseTool):
    model: ClassVar[GPT2LMHeadModel]
    tokenizer: ClassVar[GPT2Tokenizer]
class DataPreprocessingTool(BaseTool):
    name: str = "DataPreprocessingTool"
    description: str = "A tool for preprocessing and structuring unstructured data."
    name: str = "DataPreprocessingTool"
    name = "DataPreprocessingTool"
    description = "A tool for preprocessing and structuring unstructured data."
    tokenizer: GPT2Tokenizer = tokenizer
    model = model

    def _run(
        self,
        unstructured_data: Union[str, List[str]],
        run_manager: Optional[CallbackManagerForToolRun] = None
    ) -> pd.DataFrame:
        # Main function to orchestrate the preprocessing and structuring steps
        structured_data = self.structure_data(unstructured_data)
        normalized_data = self.normalize_data(structured_data)
        enriched_data = self.enrich_data(normalized_data)
        return enriched_data

    async def _arun(
        self,
        unstructured_data: Union[str, List[str]],
        run_manager: Optional[AsyncCallbackManagerForToolRun] = None
    ) -> pd.DataFrame:
        # Asynchronous version of the _run method
        # Assuming similar steps for async version
        return await self._run(unstructured_data, run_manager)

    def preprocess(self, text: str) -> str:
        # Clean and tokenize text
        cleaned_text = self.clean_text(text)
        tokens = self.tokenize(cleaned_text)
        return " ".join(tokens)

    def clean_text(self, text: str) -> str:
        # Perform general cleaning like lowercasing, punctuation removal, etc.
        cleaned_text = text.lower().replace('.', '').replace(',', '').replace('!', '').replace('?', '')
        return cleaned_text

    def tokenize(self, text: str) -> List[str]:
        # Tokenize the text using the tokenizer
        tokens = self.tokenizer.tokenize(text)
        return tokens

    def categorize_feedback(self, feedback: str) -> str:
        # Preprocess the feedback
        preprocessed_feedback = self.preprocess(feedback)
        # Tokenize the preprocessed feedback for the model
        inputs = self.tokenizer(preprocessed_feedback, return_tensors="pt")
        # Generate category using the fine-tuned model
        outputs = self.model.generate(inputs["input_ids"], max_length=50)
        category = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return category

    def structure_data(self, data: Union[str, List[str]]) -> pd.DataFrame:
        # Transforms unstructured data into a structured format
        # Assume each item in data is a text document
        structured_data_list = []
        for item in data:
            cleaned_text = self.clean_text(item)
            tokens = self.tokenize(cleaned_text)
            vector = self.vectorize_text(cleaned_text)
            structured_data_list.append({
                'cleaned_text': cleaned_text,
                'tokens': tokens,
                'vector': vector
            })
        structured_data = pd.DataFrame(structured_data_list)
        return structured_data

    def normalize_data(self, data: pd.DataFrame) -> pd.DataFrame:
        # Normalizes data (e.g., scaling features to a standard range)
        scaler = MinMaxScaler()
        # Assume data has a 'vector' column with numerical values
        data['vector'] = list(scaler.fit_transform(data['vector'].tolist()))
        return data

    def enrich_data(self, data: pd.DataFrame) -> pd.DataFrame:
        # Enriches data with additional information
        # Example: Adding a column for the number of tokens in each document
        data['num_tokens'] = data['tokens'].apply(len)
        return data

    def vectorize_text(self, text: str) -> List[float]:
        # Converts text to a numerical vector using the tokenizer
        inputs = self.tokenizer(text, return_tensors="pt")
        vector = inputs['input_ids'][0].tolist()
        return vector

# Example usage
preprocessing_tool = DataPreprocessingTool()
user_feedback = "The app crashes when I try to upload a photo."
category = preprocessing_tool.categorize_feedback(user_feedback)
print(f"Categorized as: {category}")

PydanticUserError: A non-annotated attribute was detected: `model = GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50258, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50